### Overview

Filters the TCGA-BRCA dataset to retain only the 50 PAM50 genes from the original 60,660 genes for breast cancer subtype classification.

In [1]:
import pandas as pd
import numpy as np
import re

### 1. Import data

In [2]:
# import tpm counts
tpm_counts = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/TCGA_BRCA/tcga_brca_cleaned_data/cleaned_tpm_counts.csv",
                        header=0, index_col=0)

In [3]:
# import raw counts
raw_counts = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/TCGA_BRCA/tcga_brca_cleaned_data/cleaned_raw_counts.csv",
                        header=0, index_col=0)

In [4]:
# import sample info
sample_info = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/TCGA_BRCA/tcga_brca_cleaned_data/cleaned_sample info.csv",
                        header=0, index_col=0)

In [16]:
# import PAM50 genes
pam50 = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/pam50gene_list_csv.csv", header=0)
pam50

,old_gene_symbol,new_gene_symbol,ensembl_gene_id
0,ACTR3B,ACTR3B,ENSG00000133627
1,ANLN,ANLN,ENSG00000011426
2,BAG1,BAG1,ENSG00000107262
3,BCL2,BCL2,ENSG00000171791
4,BIRC5,BIRC5,ENSG00000089685
5,BLVRA,BLVRA,ENSG00000106605
6,CCNB1,CCNB1,ENSG00000134057
7,CCNE1,CCNE1,ENSG00000105173
8,CDC20,CDC20,ENSG00000117399
9,CDC6,CDC6,ENSG00000094804


In [17]:
print(tpm_counts.shape)
print(raw_counts.shape)
print(sample_info.shape)

(1031, 60660)
(1031, 60660)
(1031, 85)


In [18]:
tpm_counts.head(3)

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000288661,ENSG00000288662,ENSG00000288663,ENSG00000288665,ENSG00000288667,ENSG00000288669,ENSG00000288670,ENSG00000288671,ENSG00000288674,ENSG00000288675
TCGA-D8-A146-01A-31R-A115-07,49.6341,9.3826,115.1737,20.1202,6.1859,10.9585,30.9154,47.9991,19.9032,56.4289,...,0.0,0.0,0.4513,0.0,0.0,0.0,10.5835,0.0,0.0411,1.1776
TCGA-AQ-A0Y5-01A-11R-A14M-07,12.0296,0.3785,134.9047,15.5758,4.3777,4.0015,29.7787,91.9948,15.5230,51.4084,...,0.0,0.0,0.1525,0.0,0.0,0.0,26.3496,0.0,0.0387,1.3672
TCGA-C8-A274-01A-11R-A16F-07,90.4249,0.0000,110.8229,31.8373,15.4869,3.3594,6.9632,33.7734,9.6288,61.4116,...,0.0,0.0,0.3309,0.0,0.0,0.0,25.0229,0.0,0.0669,0.5202


### 2. Remove version number in ENSEMBL IDs and subset PAM50 genes

In [19]:
# function to remove everything after the full stop in the ensemble gene id column names
def remove_after_full_stop(s):
    return re.sub(r'\..*', '', s)

# remove remove everything after the full stop in the ensemble gene id column names in both tpm and raw count datasets
new_col_tpm = [remove_after_full_stop(s) for s in tpm_counts.columns]
new_col_raw = [remove_after_full_stop(s) for s in raw_counts.columns]

# rename the columns in tpm and raw count datasets
tpm_counts.columns = new_col_tpm
raw_counts.columns = new_col_raw

In [20]:
# keep only PAM50 genes
tpm_pam50 = tpm_counts[pam50['ensembl_gene_id'].tolist()]
raw_pam50 = raw_counts[pam50['ensembl_gene_id'].tolist()]

print(tpm_pam50.shape)
print(raw_pam50.shape)

(1031, 50)
(1031, 50)


In [21]:
# check whether the gene column order is same as the order of ensembl gene id column in the pam50 gene df
print(tpm_pam50.columns.tolist() == pam50['ensembl_gene_id'].tolist())
print(raw_pam50.columns.tolist() == pam50['ensembl_gene_id'].tolist())

True
True


In [22]:
# check whether the PAM50 gene columns match in both tpm and raw counts datasets
print(tpm_pam50.columns.equals(raw_pam50.columns))

True


In [23]:
# check for low-quality samples (characterized by having lots of low-count genes)
# check for rows with less than 10 raw read counts in all the gene columns
# 10 is the threshold for low count genes according to DESeq2 analysis documentation
print((raw_pam50.iloc[:, 0:50] <= 10).all(axis=1).sum())

# check rows with less than 10 raw read counts in at least half of the gene columns
print(((raw_pam50.iloc[:, 0:50] <= 10).sum(axis=1) >= raw_pam50.iloc[:, 0:50].shape[1] // 2).sum())

0
0


### 3. Merge with subtype label

In [24]:
# check whether the sample indices of raw and tpm datasets match with sample info dataset
print(raw_pam50.index.equals(sample_info.index))
print(tpm_pam50.index.equals(sample_info.index))

True
True


In [25]:
# extract subtype column from sample info dataset
sample_subtype = sample_info.loc[:,['paper_BRCA_Subtype_PAM50']]

# rename column
sample_subtype.columns = ['subtype']

In [26]:
# merge raw counts and subtype 
raw_subtype = pd.merge(raw_pam50, sample_subtype, left_index=True, right_index=True) 

# merge tpm counts and subtype 
tpm_subtype = pd.merge(tpm_pam50, sample_subtype, left_index=True, right_index=True) 

In [27]:
print(raw_subtype['subtype'].tolist() == tpm_subtype['subtype'].tolist())

True


In [28]:
# sort the order of the tpm and raw count datasets based on the sample order in sample info 
raw_subtype_sorted = raw_subtype.loc[sample_subtype.index,:]
tpm_subtype_sorted = tpm_subtype.loc[sample_subtype.index,:]
print(raw_subtype_sorted.index.equals(sample_subtype.index))
print(tpm_subtype_sorted.index.equals(sample_subtype.index))

True
True


In [35]:
# # save files
# raw_subtype_sorted.to_csv('pam50genes_raw_counts_subtype_tcga_brca.csv')
# tpm_subtype_sorted.to_csv('pam50genes_tpm_counts_subtype_tcga_brca.csv')